In [0]:
select * from techdados.bronze.customers

In [0]:
--Query em SQL
CREATE OR REPLACE TABLE techdados.silver.customers
COMMENT 'Silver – customers base'
AS
SELECT
    customer_id,
    name,
    email,
    CAST(created_at AS DATE) AS created_at
FROM techdados.bronze.customers
WHERE customer_id IS NOT NULL

-- Query em pyspark
-- from pyspark.sql import functions as F

-- # Ler e transformar
-- df_customers = (
--     spark.table("techdados.bronze.customers")
--     .filter(F.col("customer_id").isNotNull())
--     .select(
--         "customer_id",
--         "name",
--         "email",
--         F.col("created_at").cast("date").alias("created_at")
--     )
-- )

-- # Salvar como tabela
-- df_customers.write \
--     .mode("overwrite") \
--     .option("overwriteSchema", "true") \
--     .option("comment", "Silver – customers base") \
--     .saveAsTable("techdados.silver.customers")

In [0]:
CREATE OR REPLACE TABLE techdados.silver.customer_addresses
COMMENT 'Silver – exploded customer addresses'
AS
SELECT
    customer_id,
    inline(addresses)
FROM techdados.bronze.customers

In [0]:
CREATE OR REPLACE TABLE techdados.silver.customer_phones
COMMENT 'Silver – exploded customer phones'
AS
SELECT
    customer_id,
    phone AS phone_number
FROM techdados.bronze.customers
LATERAL VIEW explode_outer(phones) AS phone

In [0]:
CREATE OR REPLACE TABLE techdados.silver.customer_preferences
COMMENT 'Silver – exploded customer preferences'
AS
SELECT
    customer_id,
    preferences.newsletter AS newsletter,
    preferences.sms        AS sms,
    category
FROM techdados.bronze.customers
LATERAL VIEW explode(preferences.categories) AS category

In [0]:
CREATE OR REPLACE TABLE techdados.silver.customer_purchases
COMMENT 'Silver – exploded customer purchase history'
AS
SELECT
    customer_id,
    purchase AS purchase_value
FROM techdados.bronze.customers
LATERAL VIEW explode(purchase_history) AS purchase